In [1]:
import datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
import evaluate
import torch

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


## Get Dataset

In [3]:
# Change for the name you want
DATASET_NAME = "GeoSQA"

dataset = datasets.load_dataset("rfr2003/GeoBenchLLM", DATASET_NAME)

dataset

DatasetDict({
    train: Dataset({
        features: ['question_id', 'scenario_id', 'answer', 'annotation', 'scenario', 'question', 'A', 'B', 'C', 'D'],
        num_rows: 2644
    })
    validation: Dataset({
        features: ['question_id', 'scenario_id', 'answer', 'annotation', 'scenario', 'question', 'A', 'B', 'C', 'D'],
        num_rows: 628
    })
    test: Dataset({
        features: ['question_id', 'scenario_id', 'answer', 'annotation', 'scenario', 'question', 'A', 'B', 'C', 'D'],
        num_rows: 838
    })
})

## Apply prompt

In [4]:
# You can change the prompt here. 
# If you want to use the same prompts as us, you can copy the get_message function from the good class in src/utils/datasets_handler.py
def get_messages(row):
    question = f"""Image Annotation :
    {row['annotation']}
    
    Scenario :
    {row['scenario']}

    Question :
    {row['question']}

    Choices :
    A) {row['A']}
    B) {row['B']}
    C) {row['C']}
    D) {row['D']}
    """
    messages = [
        {"role": "system", "content": "You are a strict and precise geography assistant specialized in GaoKao-style multiple-choice questions. You will receive an IMAGE TEXT ANNOTATION and a linked SCENARIO followed by a QUESTION and several answer choices labeled A, B, C, D (or similar). Exactly one option is correct. Your task is to identify the correct answer using the IMAGE ANNOTATION and the SCENARIO and reply ONLY with the corresponding letter (e.g., A, B, C, or D). Do not include any explanation, reasoning, punctuation, or extra text. Respond with a single uppercase letter only."},
        {"role": "user", "content": question}
    ]
    
    
    return messages

## Get Model and tokenizer

In [5]:
# Change for the model you want
MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype="auto",
    device_map="auto"
).to(device)

In [6]:
# Set to true if you want to enable thinking
IS_THINKING = False

## Tokenize

In [7]:
def get_prompt(row):
    messages = get_messages(row)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=IS_THINKING
    )
    return {'text': text}
    
dataset['test'] = dataset['test'].map(get_prompt)

print(dataset['test']['text'][0])

Map: 100%|██████████| 838/838 [00:00<00:00, 6259.31 examples/s]

<|im_start|>system
You are a strict and precise geography assistant specialized in GaoKao-style multiple-choice questions. You will receive an IMAGE TEXT ANNOTATION and a linked SCENARIO followed by a QUESTION and several answer choices labeled A, B, C, D (or similar). Exactly one option is correct. Your task is to identify the correct answer using the IMAGE ANNOTATION and the SCENARIO and reply ONLY with the corresponding letter (e.g., A, B, C, or D). Do not include any explanation, reasoning, punctuation, or extra text. Respond with a single uppercase letter only.<|im_end|>
<|im_start|>user
Image Annotation :
    On June 18, 2012, the Shenzhou-9 spacecraft carrying astronauts Jing Haipeng, Liu Wang, and Liu Yang achieved automated rendezvous and docking with the Tiangong-1 target spacecraft.
    
    Scenario :
    CNTV News: On June 18, 2012, the Shenzhou-9 spacecraft carrying astronauts Jing Haipeng, Liu Wang, and Liu Yang achieved automated rendezvous and docking with the Tiangong

In [8]:
# If you want to sample from the dataset. Put n_samples to None if you don't want sampling
n_samples = None

if n_samples:
    dataset['test'] = dataset['test'].shuffle(seed=42)
    if n_samples > len(dataset['test']):
        n_samples = len(dataset['test'])
    dataset['test'] = dataset['test'].select(range(n_samples))

## Generate

In [9]:
# You can change here how many new tokens you want your model to generate
MAX_TOKENS = 100

if IS_THINKING:
    MAX_TOKENS = 10000

In [10]:
generation_config = {
    'temperature': 0.7,
    'top_p': 0.8,
    'top_k': 20,
    'min_p': 0.0,
}

if IS_THINKING:
    generation_config = {
        'temperature': 0.6,
        'top_p': 0.95,
        'top_k': 20,
        'min_p': 0.0,
    }

In [11]:
generations = []

for text in tqdm(dataset['test']['text']):
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=MAX_TOKENS,
        **generation_config
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

    try:
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0
        
    #thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    generations.append(content)

100%|██████████| 838/838 [00:31<00:00, 26.58it/s]


In [12]:
print(generations[:3])
print(dataset['test']['answer'][:3])

['A', 'A', 'A']
['D', 'A', 'B']


## Evaluate

In [13]:
# Change here for the correct metric to use for your case
metric = evaluate.load("rfr2003/mcq_eval")

In [14]:
# If you want to change the dataset, you'll need to adapt these two functions.
# You can take example from the functions results and evaluate for the right dataset in src/utils/datasets_handler.py

def results(outputs, dataset):
    results = []
    for out, a in zip(outputs, dataset['answer']):
        dic = {"generated_answer":out,
               "gold_answer":a
              }
        results.append(dic)
    return results

def _evaluate(results):
    gens, golds = [], []
    for el in results:
        gens.append(el['generated_answer'])
        golds.append(el['gold_answer'])
    #change the line below if needed
    return metric.compute(generations=gens, golds=golds)

In [15]:
results = _evaluate(results(generations, dataset['test']))

In [16]:
print(results)

{'accuracy': 0.21241050119331742}
